# CV15 — Dự đoán frame tương lai

CNN mã hóa mỗi frame, GRU tích lũy lịch sử, decoder dựng frame tiếp theo. Loss so output ở thời điểm t với frame t+1. Ví dụ teacher forcing, chưa có action hoặc rollout autoregressive.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

CNN mã hóa mỗi frame, GRU tích lũy lịch sử, decoder dựng frame tiếp theo. Loss so output ở thời điểm t với frame t+1. Ví dụ teacher forcing, chưa có action hoặc rollout autoregressive. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv15_world import World, loss_fn
video = torch.rand(2, 4, 3, 32, 32, device=device)
model = World(d=32).to(device)
batch = (video,)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
model.eval()
with torch.no_grad():
    prediction = model(video[:, :-1])
print('Predicted future frames:', prediction.shape)
print('Aligned targets:', video[:, 1:].shape)
print('Repeat-last baseline L1:', F.l1_loss(video[:, :-1], video[:, 1:]).item())


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Repeat-last → CNN+GRU → spatial latent dynamics → action-conditioned model. Global average pooling làm mất cấu trúc không gian, là điểm cần nâng cấp.


In [ ]:
model.eval()
with torch.no_grad():
    prediction = model(video[:, :-1])
print('Fixture one-step L1:', F.l1_loss(prediction, video[:, 1:]).item())
print('A real study must also measure multi-step rollout stability and split by original video.')


## Thử nghiệm tiếp theo

Repeat-last → CNN+GRU → spatial latent dynamics → action-conditioned model. Global average pooling làm mất cấu trúc không gian, là điểm cần nâng cấp.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv15_world.py) và [tài liệu gốc](../REFERENCE.md).
